In [1]:
# Fail-safe: Re-create 'df' if it's missing
import pandas as pd

if 'df' not in globals():
    print("df not found. Re-merging from source files...")
    # Update these paths if your files are in a different folder!
    app_df = pd.read_csv(r'..\creditcard_csv\application_record.csv')
    credit_df = pd.read_csv(r'..\creditcard_csv\credit_record.csv')
    
    # Re-apply the aggregation logic
    high_risk_statuses = ['2', '3', '4', '5']
    credit_df['Risk_Flag'] = credit_df['STATUS'].isin(high_risk_statuses).astype(int)
    credit_agg = credit_df.groupby('ID')['Risk_Flag'].max().reset_index()
    credit_agg.rename(columns={'Risk_Flag': 'Is_High_Risk'}, inplace=True)
    
    # Re-merge
    df = pd.merge(app_df, credit_agg, on='ID', how='inner')
    
    # Re-apply basic cleaning (strip columns)
    df.columns = df.columns.str.strip()
    
    print("df has been recreated successfully!")
else:
    print("df is already in memory. You are good to go!")

df not found. Re-merging from source files...
df has been recreated successfully!


In [2]:
# --- ENCODING LOCKDOWN (Run this before Model Building) ---

# 1. Identify any remaining columns that are not numbers
object_cols = df.select_dtypes(include=['object']).columns.tolist()

if len(object_cols) > 0:
    print(f"Found non-numeric columns: {object_cols}. Encoding now...")
    # Convert all object columns to binary (0/1) columns using One-Hot Encoding
    df = pd.get_dummies(df, columns=object_cols, drop_first=True, dtype=int)
    print("Encoding complete. All data is now numeric.")
else:
    print("All data is already numeric. Proceeding to Model Building...")

# 2. Re-define X and y AFTER the encoding is finalized
X = df.drop('Is_High_Risk', axis=1)
y = df['Is_High_Risk']

print("X and y updated. You are now ready to run the Model Building cell.")

Found non-numeric columns: ['CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE']. Encoding now...
Encoding complete. All data is now numeric.
X and y updated. You are now ready to run the Model Building cell.


In [3]:
# ==========================================
# Epic 4: Model Building Setup
# ==========================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# NEW: Import SMOTE for the severe class imbalance in credit approvals
# (Make sure you ran '%pip install imbalanced-learn' earlier!)
from imblearn.over_sampling import SMOTE

print("--- Preparing Data for Modeling ---")

# Verify that our preprocessed dataframe is in memory. 
# We DO NOT load from CSV here, otherwise we lose all our feature engineering!
if 'df' not in globals():
    raise NameError("Dataframe 'df' not found! Please run the previous preprocessing cells first.")

# Separate features (X) and target (y)
# Updated target variable to 'Is_High_Risk'
X = df.drop('Is_High_Risk', axis=1)
y = df['Is_High_Risk']

# Split the data into 80% training and 20% testing
# stratify=y ensures the tiny percentage of high-risk cases is split evenly between train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Original Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

# --- NEW: Handle Class Imbalance ---
# Apply SMOTE to the training data ONLY (to prevent data leakage into the test set)
print("\nApplying SMOTE to balance the training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Resampled Training features shape: {X_train_resampled.shape}")
print(f"Balanced Target Distribution:\n{y_train_resampled.value_counts()}")

# Helper function to evaluate models so we don't repeat code
def evaluate_model(model_name, y_true, y_pred):
    print(f"\n================ {model_name} ================\n")
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))

--- Preparing Data for Modeling ---
Original Training features shape: (29165, 47)
Testing features shape: (7292, 47)

Applying SMOTE to balance the training data...
Resampled Training features shape: (57344, 47)
Balanced Target Distribution:
Is_High_Risk
0    28672
1    28672
Name: count, dtype: int64


In [4]:
print("Training Logistic Regression Model on Resampled Data...")

# Use the balanced/resampled training data instead of the raw X_train
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_resampled, y_train_resampled)

# Predict and Evaluate using the original, clean test set
y_pred_log = log_reg.predict(X_test)

# Use the helper function defined in your setup block
evaluate_model("Logistic Regression", y_test, y_pred_log)

Training Logistic Regression Model on Resampled Data...

================ Logistic Regression ================

Confusion Matrix:
[[5698 1471]
 [  94   29]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.79      0.88      7169
           1       0.02      0.24      0.04       123

    accuracy                           0.79      7292
   macro avg       0.50      0.52      0.46      7292
weighted avg       0.97      0.79      0.87      7292



In [5]:
# Story 2: Decision Tree Model
print("Training Decision Tree Model...")

# Initialize and train the model
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

# Predict and Evaluate
y_pred_tree = tree_model.predict(X_test)
evaluate_model("Decision Tree", y_test, y_pred_tree)

Training Decision Tree Model...

================ Decision Tree ================

Confusion Matrix:
[[7092   77]
 [  96   27]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7169
           1       0.26      0.22      0.24       123

    accuracy                           0.98      7292
   macro avg       0.62      0.60      0.61      7292
weighted avg       0.97      0.98      0.98      7292



In [6]:
# Story 3: Random Forest Model
print("Training Random Forest Model on Resampled Data...")

# Initialize Random Forest
# n_estimators=100 is standard; n_jobs=-1 uses all CPU cores for speed
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Fit on the SMOTE-balanced training data
rf_model.fit(X_train_resampled, y_train_resampled)

# Predict and Evaluate using the original, clean test set
y_pred_rf = rf_model.predict(X_test)

# Use the helper function defined in your setup block
evaluate_model("Random Forest", y_test, y_pred_rf)

Training Random Forest Model on Resampled Data...

================ Random Forest ================

Confusion Matrix:
[[7103   66]
 [ 102   21]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7169
           1       0.24      0.17      0.20       123

    accuracy                           0.98      7292
   macro avg       0.61      0.58      0.59      7292
weighted avg       0.97      0.98      0.98      7292



In [7]:
import pickle
import os

# Assuming 'rf_model' is already trained from the previous step 
# and 'X_train_resampled' was used to fit it.

print("5. Saving the model directly to your project folder...")

# We save as 'approval_model.pkl' to distinguish it from the old fraud one
target_path = r'..\approval_model.pkl'

try:
    with open(target_path, 'wb') as file:
        pickle.dump(rf_model, file)
    
    # Verify it worked
    if os.path.exists(target_path):
        print(f"\n✅ SUCCESS! The model is saved exactly at: {target_path}")
        print("You can now open your terminal and run your application!")
    else:
        print("\n❌ FAILED: Something blocked Python from saving the file.")

except Exception as e:
    print(f"\n❌ ERROR: Could not save the model: {e}")

5. Saving the model directly to your project folder...

✅ SUCCESS! The model is saved exactly at: ..\approval_model.pkl
You can now open your terminal and run your application!
